# 1.3.1. Neuroninis tinklas banko klientų duomenims

**Liutauras Turauskis · DIPf-24 · ND1 tęsinys**

Pritaikyta pagal dėstytojo pateiktą `AdventureWorksDW_NN.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Dviračio pirkimo prognozė pakeista indėlio užsakymo prognoze; išsaugotas skaitinių ir kategorinių požymių apdorojimas, tankieji sluoksniai, mokymas ir išsaugojimas.

**Paleidimas:** Google Colab → „Runtime“ → „Run all“. Ląsteles vykdyti nuo pradžios.
Duomenys atsisiunčiami automatiškai; savo kodo, failų ir API rakto įterpti nereikia.
GPU rekomenduojamas vaizdų mokymui, bet nėra būtinas. Pirmas atsisiuntimas gali užtrukti.

**Rezultatų sąžiningumas:** šioje pateikiamoje kopijoje mokymo išvestys išvalytos.
Rodikliai, grafikai ir konkrečios išvados apskaičiuojami paleidus, o ne įrašomi iš anksto.
Parengimo aplinkoje TensorFlow mokymas nebuvo vykdytas; prieš pateikiant reikia paleisti visą užrašyną
Colab ir išsaugoti kopiją su išvestimis. Atsitiktinė sėkla mažina rezultatų svyravimą, tačiau GPU
ir bibliotekų versijos gali lemti nedidelius skirtumus.


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 1. Kiti verslo duomenys ir prognozės tikslas

Originale buvo „AdventureWorks“ klientai ir `BikeBuyer` (ar pirko dviratį).
Čia naudojami **UCI Bank Marketing** duomenys: ar klientas užsisakė terminuotąjį indėlį (`y`).
Tai viešas pasirinktas, o ne mano paties surinktas rinkinys. Originalūs autoriai:
S. Moro, P. Rita, P. Cortez; [UCI šaltinis ir CC BY 4.0 licencija](https://archive.ics.uci.edu/dataset/222/bank+marketing).
Verslo rinkinys taip pat platinamas [Kaggle](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset);
čia naudojamas UCI originalas, kad nereikėtų Kaggle prisijungimo.

Pasirenkama iki 8 000 įrašų iš `bank-full.csv`, išlaikant klasių proporcijas.
`duration` pašalinamas: skambučio trukmė nežinoma prieš skambutį ir lemtų informacijos nutekėjimą.
Likę kontaktų bei ankstesnės kampanijos požymiai naudojami kaip turima kliento informacija.
Tai mokomasis atsitiktinio padalijimo eksperimentas; būsimos kampanijos kokybei reikėtų laikinio padalijimo.


In [ ]:
URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
r = requests.get(URL, timeout=90)
r.raise_for_status()
# UCI archyve bank-full.csv gali būti papildomame bank.zip archyve.
def read_bank_csv(blob):
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        names = archive.namelist()
        csv_name = next((n for n in names if n.endswith('bank-full.csv')), None)
        if csv_name:
            return pd.read_csv(archive.open(csv_name), sep=';')
        nested = next((n for n in names if n.rsplit('/', 1)[-1] == 'bank.zip'), None)
        if nested:
            return read_bank_csv(archive.read(nested))
    raise ValueError('UCI archyve nerastas bank-full.csv; patikrinkite šaltinį.')

raw = read_bank_csv(r.content)
assert set(raw['y']) == {'yes', 'no'}
df = raw.drop(columns=['duration']).copy()
df['target'] = df.pop('y').map({'no': 0, 'yes': 1}).astype('int32')
if len(df) > 8000:
    df, _ = train_test_split(df, train_size=8000, stratify=df['target'], random_state=SEED)
train, remaining = train_test_split(df, test_size=0.30, stratify=df['target'], random_state=SEED)
val, test = train_test_split(remaining, test_size=0.50, stratify=remaining['target'], random_state=SEED)
assert set(train.index).isdisjoint(val.index) and set(train.index).isdisjoint(test.index)
assert set(val.index).isdisjoint(test.index)
assert 'target' not in train.drop(columns='target') and 'duration' not in df
display(df.head())
display(pd.DataFrame({k: {'įrašai': len(v), 'indėlis_1_dalis': v.target.mean()}
                      for k, v in {'mokymas': train, 'validavimas': val, 'testas': test}.items()}).T)
display(df.dtypes.rename('tipas').to_frame())
print('Trūkstamos reikšmės:', int(df.isna().sum().sum()))
# unknown paliekamas kaip atskira kategorija; pdays=-1 reiškia, kad anksčiau nekontaktuota.
assert not df.isna().any().any()
for name, part in [('train', train), ('val', val), ('test', test)]:
    part.to_csv(OUT / f'bank_{name}.csv', index=False)


## 2. Duomenų paruošimas tinklui

Originalo `df_to_dataset` turėjo riziką palikti tikslą požymių žodyne. Čia `target` pirmiausia
atskiriamas ir į modelio įvestis nepatenka. Skaitinius stulpelius standartizuoja `Normalization`,
tekstines kategorijas `StringLookup` paverčia one-hot vektoriais. Abiejų sluoksnių `adapt`
mato **tik mokymo** duomenis; validavimo bei testo statistika nenaudojama.
Nežinoma būsimo įrašo kategorija turi atskirą OOV vietą.


In [ ]:
numeric = train.drop(columns='target').select_dtypes(include='number').columns.tolist()
categorical = train.drop(columns='target').select_dtypes(exclude='number').columns.tolist()

def feature_dict(frame):
    features = frame.drop(columns='target', errors='ignore')
    assert 'target' not in features
    return {c: tf.convert_to_tensor(features[c].to_numpy(dtype='float32' if c in numeric else str).reshape(-1, 1),
                                    dtype=tf.float32 if c in numeric else tf.string)
            for c in features.columns}

def as_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((feature_dict(frame), frame['target'].to_numpy('float32')))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)

inputs, encoded = {}, []
for column in numeric:
    inp = keras.Input(shape=(1,), name=column, dtype='float32')
    normalizer = layers.Normalization(axis=None)
    normalizer.adapt(train[column].to_numpy('float32'))
    inputs[column] = inp
    encoded.append(normalizer(inp))
for column in categorical:
    inp = keras.Input(shape=(1,), name=column, dtype='string')
    lookup = layers.StringLookup(output_mode='one_hot')
    lookup.adapt(tf.convert_to_tensor(train[column].to_numpy(str), dtype=tf.string))
    inputs[column] = inp
    encoded.append(lookup(inp))
print('Skaitiniai:', numeric)
print('Kategoriniai:', categorical)
train_ds, val_ds, test_ds = as_dataset(train, True), as_dataset(val), as_dataset(test)


## 3. Modelio sandara ir mokymas

Kaip originale, sujungti požymiai pereina per 64 ir 32 neuronų ReLU sluoksnius bei `Dropout`.
Vienas sigmoid išėjimas reiškia klasės „užsisakė indėlį“ tikimybės įvertį. Jam tinka įprastas
`binary_crossentropy`; nenaudojamas logits režimas. `EarlyStopping` pagal validavimo nuostolį
atkuria geriausius svorius. Retesnei teigiamai klasei mokant suteikiamas didesnis svoris.


In [ ]:
x = layers.Concatenate()(encoded)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dropout(0.3)(x)
model = keras.Model(inputs, layers.Dense(1, activation='sigmoid')(x))
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.AUC(name='auc')])
model.summary()
counts = train.target.value_counts()
weights = {int(k): len(train) / (2 * int(n)) for k, n in counts.items()}
history = model.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=weights,
    callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
               keras.callbacks.TensorBoard(log_dir=str(OUT / 'tensorboard_nn'))], verbose=2)
pd.DataFrame(history.history).to_csv(OUT / 'nn_mokymo_eiga.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric in zip(axes, ['loss', 'auc']):
    ax.plot(history.history[metric], label='mokymas')
    ax.plot(history.history['val_' + metric], label='validavimas')
    ax.set(xlabel='Epocha (nuo 0)', title=metric)
    ax.legend()
plt.tight_layout(); plt.show()


## 4. Nepriklausomas testas, etalonas ir išsaugojimas

Kadangi klasių skaičiai skiriasi, vien accuracy neužtenka: rodomi precision, recall, F1,
ROC AUC ir average precision (PR kreivės santrauka). Paprastas etalonas visiems priskiria
mokymo daugumos klasę. F1 slenkstis parenkamas validavime; testas naudojamas tik galutiniam vertinimui.
`.keras` faile saugomas ir požymių apdorojimas. Tikriname, ar įkeltas modelis grąžina tuos pačius rezultatus.


In [ ]:
def binary_metrics(y, probabilities, threshold):
    predicted = (np.asarray(probabilities).ravel() >= threshold).astype(int)
    return {'accuracy': accuracy_score(y, predicted),
            'precision': precision_score(y, predicted, zero_division=0),
            'recall': recall_score(y, predicted, zero_division=0),
            'f1': f1_score(y, predicted, zero_division=0),
            'roc_auc': roc_auc_score(y, probabilities),
            'pr_auc_AP': average_precision_score(y, probabilities)}

def choose_threshold(y, probabilities):
    # Slenkstis pasirenkamas TIK validavimo rinkinyje. Testo rodikliai jo nekeičia.
    thresholds = np.linspace(0.05, 0.95, 91)
    scores = [f1_score(y, np.asarray(probabilities).ravel() >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(scores))])


In [ ]:
p_val = model.predict(feature_dict(val), verbose=0).ravel()
threshold = choose_threshold(val.target, p_val)
p_test = model.predict(feature_dict(test), verbose=0).ravel()
metrics = binary_metrics(test.target, p_test, threshold)
majority = int(train.target.mode().iloc[0])
baseline = binary_metrics(test.target, np.full(len(test), float(majority)), 0.5)
display(pd.DataFrame([baseline, metrics], index=['daugumos_etalonas', 'neuroninis_tinklas']))
ConfusionMatrixDisplay.from_predictions(test.target, p_test >= threshold,
    display_labels=['ne', 'taip'], cmap='Blues')
plt.title(f'Testas; slenkstis iš validavimo = {threshold:.2f}'); plt.show()
model.save(OUT / 'bank_nn.keras')
restored = keras.models.load_model(OUT / 'bank_nn.keras')
np.testing.assert_allclose(p_test, restored.predict(feature_dict(test), verbose=0).ravel(), atol=1e-5)
(OUT / 'nn_rodikliai.json').write_text(json.dumps({'threshold': threshold, **metrics}, indent=2))
sample = test.drop(columns='target').iloc[[0]]
display(sample)
print('Tikimybės įvertis:', float(restored.predict(feature_dict(sample), verbose=0)[0, 0]),
      '| Tikroji klasė:', int(test.target.iloc[0]))
print(f'Testo F1={metrics["f1"]:.3f}; AP={metrics["pr_auc_AP"]:.3f}; '
      f'etaloninis accuracy={baseline["accuracy"]:.3f}.')
print('Įkėlus modelį prognozės sutapo. Rodikliai apibūdina tik šį duomenų padalijimą.')


## 5. Paaiškinimas ir ribos

Požymiai → skaitinimas ir normalizavimas → neuroninis tinklas → indėlio tikimybės įvertis.
Mažas validavimo nuostolis pats savaime negarantuoja geros recall ar kalibruotų tikimybių.
Grafike didėjantis atotrūkis tarp mokymo ir validavimo gali rodyti persimokymą.
F1 slenkstis yra mokomoji taisyklė: realiame banke jį reikėtų sieti su kampanijos kaštais.
Antrame 1.3 faile tas pats duomenų padalijimas leidžia nagrinėti alternatyvų algoritmą.

Papildoma dokumentacija: [TensorFlow struktūrinių duomenų klasifikavimas](https://www.tensorflow.org/tutorials/structured_data/preprocessing_layers).
